# What Is Web Scraping?

Web scraping is the process of automatically collecting information from websites.
Instead of copying and pasting text manually, a scraper downloads the page’s HTML and extracts the parts you care about—articles, prices, tables, links, images, or structured data.

Web scraping is widely used for:
- **Research** — collecting articles, papers, or public datasets
- **Business intelligence** — tracking prices, competitors, or product details
- **Data analysis** — gathering tables, statistics, or structured information
- **Automation** — pulling repeated information without manual effort

At its core, web scraping is simply:
1. **Download HTML**
2. **Parse HTML**
3. **Extract the data you want**
4. **Clean and store it**

---

# Static vs Dynamic Websites

### Static websites
These return complete HTML directly from the server.
Examples: documentation pages, blogs, Wikipedia.

Static pages are **easy** to scrape because the HTML already contains the data.

### Dynamic websites
These use JavaScript to load data after the page appears.
Examples: modern dashboards, infinite-scroll feeds, interactive apps.

Dynamic pages often require:
- API calls
- Browser automation (Selenium, Playwright)
- Network inspection tools

For beginners, we focus on **static HTML**, which BeautifulSoup handles perfectly.

---

# Why BeautifulSoup?

BeautifulSoup (BS4) is a Python library that turns messy HTML into a structured, searchable tree.

It is ideal for teaching because:
- **Beginner-friendly API**
- **Handles imperfect HTML gracefully**
- **Works seamlessly with `requests`**
- **Provides simple functions for finding elements**
- **Allows CSS selectors, tag searches, and attribute filtering**

The core workflow is:

```python
from bs4 import BeautifulSoup
import requests

html = requests.get(url).text
soup = BeautifulSoup(html, "html.parser")
```

Once you have a `soup` object, you can navigate the page using:
- `find()` — first matching element
- `find_all()` — all matching elements
- `select()` — CSS selectors
- `select_one()` — first CSS selector match
- `.get_text()` — extract text
- `.attrs` — access attributes
- Parent/child/sibling navigation

These are the tools students will use 95% of the time.

---

# What This Notebook Covers

This notebook teaches:
1. **How to load HTML into BeautifulSoup**
2. **How to search for elements**
3. **How to extract text and attributes**
4. **How to navigate nested structures**
5. **How to scrape real-world pages**
6. **How to clean and structure extracted data**

It is designed as a **beginner-friendly, hybrid lecture + lab**, with clear examples and reusable patterns.


# BeautifulSoup Module
This notebook introduces **BeautifulSoup**, a Python library that makes it easy to read and work with HTML. Webpages are often messy or inconsistent, and BeautifulSoup helps you:
- load HTML safely
- explore the structure of a webpage
- find specific tags
- extract text or attributes
- navigate nested elements
- modify HTML content

Each section includes small, focused examples designed for beginners learning web scraping.

## ANSI Color Codes
These variables store color codes used to make printed output easier to read. They do not affect scraping logic — they simply colorize text so you can visually distinguish different parts of the output.

In [ ]:
red = '\033[1;91m'
yellow = '\033[1;93m'
green = '\033[1;92m'
cyan = '\033[1;96m'
blue = '\033[1;94m'
purple = '\033[1;95m'
white = '\033[1;97m'
black = '\033[1;90m'
orange = '\033[38;5;208m'
lavender = '\033[38;5;183m'
gray = '\033[38;5;245m'
NAK = white

## Imports
We import:
- **BeautifulSoup** for parsing HTML
- **requests** for downloading webpages
- **RequestException** for catching network errors

These are the core tools used in most beginner web‑scraping projects.

In [ ]:
from bs4 import BeautifulSoup
import requests
from requests.exceptions import RequestException

## Utility: Consistent Header Printing
Every function prints a small header showing:
- the function name
- the parameters passed in

This makes the output easier to follow when running many examples in sequence.

In [ ]:
def printHeader(funcname, *parameters):
    print(f"\n{white}function: {funcname}")
    for item in parameters:
        print(f"{yellow}parameter: {item}")

# 1. Robust HTTP Fetcher
Before scraping a webpage, you must download its HTML. A simple `requests.get(url)` works sometimes, but beginners quickly run into problems:
- the site may be down
- the URL may be wrong
- the server may return an error
- the connection may time out

This function wraps `requests.get()` in a safe way:
- sets a timeout
- disables SSL verification (helpful for older sites)
- checks for HTTP errors
- catches any network exception
- returns the HTML only if successful
- returns `None` if something goes wrong

This prevents your scraper from crashing and teaches defensive programming early.

In [ ]:
def fetch_url(url, timeout=10):
    printHeader("fetch_url", url)
    try:
        response = requests.get(url, verify=False, timeout=timeout)
        response.raise_for_status()
        return response.text
    except RequestException as e:
        print(f"{red}HTTP error: {e}")
        return None

# 2. Safe Soup Creation
BeautifulSoup needs HTML input — either from a string or a file. This function creates a soup object safely by:
- reading from a file if requested
- catching parsing errors
- returning `None` if something goes wrong

Beginners often break BeautifulSoup by giving it invalid HTML, so this teaches defensive programming early.

In [ ]:
def make_soup(source, from_file=False):
    try:
        if from_file:
            with open(source, "r", encoding="utf-8") as handle:
                return BeautifulSoup(handle.read(), "html.parser")
        return BeautifulSoup(source, "html.parser")
    except Exception as e:
        print(f"{red}Parse error: {e}")
        return None

# 3. Basic HTML Parsing
This example shows how to:
- create a soup object
- safely access a specific tag (`<h1>`)
- print its text

It demonstrates the simplest possible HTML extraction and introduces the idea that tags may or may not exist.

In [ ]:
def html_parsing(html):
    printHeader("html_parsing", html)
    soup = make_soup(html)
    if soup and soup.h1:
        print(f"{red}{soup.h1.text}")
    else:
        print(f"{red}<h1 not found>")
    return soup

html1 = "<html><body><h1>Hello, World!</h1></body></html>"
html_parsing(html1)

# 4. Scrape a Web Page End-to-End
This combines two earlier ideas:
1. **download the webpage**
2. **parse the HTML with BeautifulSoup**

Students see the full workflow:
- fetch → parse → inspect

It prints the entire soup so beginners can see what real HTML looks like.

In [ ]:
def scrape_web_page(url):
    printHeader("scrape_web_page", url)
    html = fetch_url(url)
    if html is None:
        return None
    soup = make_soup(html)
    print(f"{cyan}{soup}")
    return soup

scrape_web_page("https://example.com")

# 5. Parse HTML File
Web scraping isn't only about live websites — sometimes you scrape saved HTML files.

This example:
- writes a small HTML file
- loads it from disk
- parses it with BeautifulSoup
- prints the result

It teaches students how to work with local HTML content.

In [ ]:
def parse_html_file(path):
    printHeader("parse_html_file", path)
    soup = make_soup(path, from_file=True)
    if soup:
        print(f"{green}{soup}")
    return soup

with open("file.html", "w", encoding="utf-8") as f:
    f.write("<html><body><p>File Content</p></body></html>")

parse_html_file("file.html")

# 6. Find a Single Tag
This example introduces `.find()`, one of the most common BeautifulSoup methods.

Students learn:
- how to locate the first occurrence of a tag
- how to extract its text
- how to handle the case where the tag doesn't exist

In [ ]:
def find_tag(html, tag):
    printHeader("find_tag", html, tag)
    soup = make_soup(html)
    if soup:
        element = soup.find(tag)
        print(f"{cyan}{element.text if element else '<not found>'}")

html3 = "<html><body><h1>Hello</h1><p>Paragraph</p></body></html>"
find_tag(html3, "p")

# 7. Find Multiple Tags
This example introduces `.find_all()`, which returns **all** matching tags.

Students learn how to:
- loop through multiple elements
- extract text from each one
- handle cases where tags may not exist

In [ ]:
def find_multiple_tags(html, *tags):
    printHeader("find_multiple_tags", html, *tags)
    soup = make_soup(html)
    if not soup:
        return
    for tag in tags:
        for element in soup.find_all(tag):
            print(f"{green}{element.text}")

html4 = "<html><body><p>First</p><p>Second</p></body></html>"
find_multiple_tags(html4, "p")

# 8. Extract Attributes Safely
HTML tags often contain useful attributes, such as:
- `href` in `<a>` links
- `src` in `<img>` tags
- `class` or `id` for styling

This example shows how to:
- locate a tag
- check whether it has a specific attribute
- extract and print that attribute safely

In [ ]:
def extract_attributes(html, tag, attr):
    printHeader("extract_attributes", html, tag, attr)
    soup = make_soup(html)
    if not soup:
        return
    element = soup.find(tag)
    if element and element.has_attr(attr):
        print(f"{purple}{element[attr]}")
    else:
        print(f"{purple}<attribute not found>")

html5 = '<html><body><a href="https://example.com">Click Here</a></body></html>'
extract_attributes(html5, "a", "href")

# 9. Navigate DOM Safely
HTML is structured like a tree. Tags can contain other tags, and BeautifulSoup lets you navigate through them.

This example shows how to safely access nested tags like:
```
<div>
  <p>Nested</p>
</div>
```
It also demonstrates how to handle missing elements without crashing.

In [ ]:
def navigate_dom_tree(html):
    printHeader("navigate_dom_tree", html)
    soup = make_soup(html)
    try:
        print(f"{red}{soup.div.p.text}")
    except AttributeError:
        print(f"{red}<div><p> not found>")

html6 = "<html><body><div><p>Nested</p></div></body></html>"
navigate_dom_tree(html6)

# 10. CSS Selectors
CSS selectors let you find elements using patterns like:
- `.classname`
- `#idname`
- `div p`

This example introduces `.select_one()`, which works similarly to CSS selectors used in web development.

In [ ]:
def css_selectors(html, selector):
    printHeader("css_selectors", html, selector)
    soup = make_soup(html)
    if not soup:
        return
    element = soup.select_one(selector)
    print(f"{purple}{element.text if element else '<not found>'}")

html7 = '<html><body><div class="myclass">Content</div></body></html>'
css_selectors(html7, ".myclass")

# 11. Modify HTML Content
BeautifulSoup doesn't just read HTML — it can also modify it.

This example shows how to:
- locate a tag
- replace its text
- print the updated HTML

This is useful when cleaning or transforming scraped data.

In [ ]:
def modify_html_content(html, replacement):
    printHeader("modify_html_content", html, replacement)
    soup = make_soup(html)
    if soup and soup.p:
        soup.p.string = replacement
        print(f"{green}{soup}")
    else:
        print(f"{green}<p not found>")

html8 = "<html><body><p>Old Text</p></body></html>"
modify_html_content(html8, "New Text")

# 12. Main Demonstration
This final section runs **all the examples in the notebook** in a single sequence. The goal is to show how each function behaves when executed together, so beginners can see the full workflow of:

1. downloading HTML
2. creating a BeautifulSoup object
3. finding tags
4. extracting attributes
5. navigating nested elements
6. using CSS selectors
7. modifying HTML

The `main()` function prints clear headers for each step and demonstrates how the individual pieces fit together into a complete, working web‑scraping pipeline.

Running this cell gives students a "big picture" view of how BeautifulSoup is used in practice.

In [ ]:
def main():
    print("\n=== Running BeautifulSoup Examples ===")

    html1 = "<html><body><h1>Hello, World!</h1></body></html>"
    html_parsing(html1)

    url = "https://example.com"
    soup = scrape_web_page(url)
    if soup:
        print(soup.prettify())

    html2 = "<html><body><p>File Content</p></body></html>"
    with open("file.html", "w", encoding="utf-8") as f:
        f.write(html2)
    parsed = parse_html_file("file.html")
    if parsed:
        print(parsed.prettify())

    html3 = "<html><body><h1>Hello</h1><p>Paragraph</p></body></html>"
    find_tag(html3, "p")

    html4 = "<html><body><p>First</p><p>Second</p></body></html>"
    find_multiple_tags(html4, "p")

    html5 = '<html><body><a href="https://example.com">Click Here</a></body></html>'
    extract_attributes(html5, "a", "href")

    html6 = "<html><body><div><p>Nested</p></div></body></html>"
    navigate_dom_tree(html6)

    html7 = '<html><body><div class="myclass">Content</div></body></html>'
    css_selectors(html7, ".myclass")

    html8 = "<html><body><p>Old Text</p></body></html>"
    modify_html_content(html8, "New Text")

    print("\n=== All examples completed ===")

main()